# 2. Datenaufbereitung und Integration

## Ziel

Wir bereiten die gespeicherten ElCom- und BFS-Daten auf und
verknüpfen sie über die BFS-Gemeindenummer.

Dabei dokumentieren wir:
- die Prüfung und Vereinheitlichung der Datentypen,
- fehlende Werte und doppelte Schlüssel,
- erfolgreich und nicht erfolgreich zugeordnete Datensätze,
- begründete Ausschlüsse,
- die Anzahl der Datensätze vor und nach der Verknüpfung.

Die gespeicherten Rohdaten bleiben unverändert.

In [1]:
from pathlib import Path
import json
import pandas as pd

# Projektordner bestimmen.
arbeitsordner = Path.cwd()

projektordner = (
    arbeitsordner.parent
    if arbeitsordner.name == "notebooks"
    else arbeitsordner
)

rohdaten_ordner = projektordner / "data" / "raw"

print("Rohdatenordner:", rohdaten_ordner)
print("Ordner vorhanden:", rohdaten_ordner.is_dir())

Rohdatenordner: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/raw
Ordner vorhanden: True


In [2]:
elcom_dateien = list(
    rohdaten_ordner.glob("elcom_2026_h4_*.json")
)

# Metadatendateien ausschliessen.
elcom_dateien = [
    datei
    for datei in elcom_dateien
    if not datei.name.endswith("_metadaten.json")
]

bfs_dateien = list(
    rohdaten_ordner.glob("bfs_gemeinden_2026_*.csv")
)

# Nur automatisch auswählen, wenn genau ein Abruf vorliegt.
if len(elcom_dateien) != 1 or len(bfs_dateien) != 1:
    raise ValueError(
        "Es muss genau eine ElCom-Rohdatei und eine BFS-CSV "
        "vorliegen. Bitte die verfügbaren Dateien prüfen."
    )

elcom_datei = elcom_dateien[0]
bfs_datei = bfs_dateien[0]

print("ElCom-Datei:", elcom_datei.name)
print("BFS-Datei:", bfs_datei.name)

ElCom-Datei: elcom_2026_h4_20261117T195442966207Z.json
BFS-Datei: bfs_gemeinden_2026_20261117T195841888739Z.csv


In [3]:
# ElCom-Rohantwort laden.
with elcom_datei.open(encoding="utf-8") as datei:
    elcom_raw = json.load(datei)

# Verschachtelte Antwort in Tabellenzeilen umwandeln.
rows = [
    {
        field: entry["value"]
        for field, entry in record.items()
    }
    for record in elcom_raw["results"]["bindings"]
]

df_elcom = pd.DataFrame(
    rows,
    columns=["gemeinde", "betreiber", "gesamttarif"]
)

# BFS-Rohdatei laden.
df_bfs = pd.read_csv(
    bfs_datei,
    encoding="utf-8-sig"
)

print("ElCom-Tarifzeilen:", len(df_elcom))
print("BFS-Gemeinden:", len(df_bfs))

ElCom-Tarifzeilen: 2268
BFS-Gemeinden: 2110


## 2.1 Tabellen für die Verknüpfung vorbereiten

Die Gemeindenummer liegt in den Quellen unterschiedlich vor:
- ElCom: als Bestandteil einer URI.
- BFS: in der Spalte BfsCode.

Wir extrahieren die ElCom-Gemeindenummer und verwenden in beiden
Tabellen den gleichen Spaltennamen und Datentyp.

Ausserdem wandeln wir den ElCom-Gesamttarif in einen numerischen
Datentyp um und ergänzen lesbare Bezeichnungen der Gemeindetypen.

Wir arbeiten mit Kopien. Die Rohdaten bleiben unverändert.
Es werden noch keine Zeilen ausgeschlossen.

In [4]:
df_elcom_clean = df_elcom.copy()

# Gemeindenummer aus dem Ende der URI extrahieren.
df_elcom_clean["gemeindenummer"] = (
    df_elcom_clean["gemeinde"]
    .str.extract(r"/municipality/(\d+)$", expand=False)
    .astype("Int64")
)

# Tarifwerte von Text in Zahlen umwandeln.
df_elcom_clean["gesamttarif"] = pd.to_numeric(
    df_elcom_clean["gesamttarif"],
    errors="raise"
)

df_elcom_clean.head()

,gemeinde,betreiber,gesamttarif,gemeindenummer
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,1
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,10
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,100
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1001
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1002


In [5]:
# Benötigte Felder auswählen und verständlich benennen.
df_bfs_clean = (
    df_bfs[
        ["BfsCode", "Name", "CantonId", "Canton", "STALAN2020"]
    ]
    .rename(
        columns={
            "BfsCode": "gemeindenummer",
            "Name": "gemeindename",
            "CantonId": "kanton_id",
            "Canton": "kanton",
            "STALAN2020": "gemeindetyp_code"
        }
    )
    .copy()
)

# Gleicher Schlüssel-Datentyp wie bei ElCom.
df_bfs_clean["gemeindenummer"] = (
    df_bfs_clean["gemeindenummer"].astype("Int64")
)

# Lesbare Bezeichnungen ergänzen.
typ_bezeichnungen = {
    1: "Städtisch",
    2: "Intermediär",
    3: "Ländlich"
}

df_bfs_clean["gemeindetyp"] = (
    df_bfs_clean["gemeindetyp_code"].map(typ_bezeichnungen)
)

df_bfs_clean.head()

,gemeindenummer,gemeindename,kanton_id,kanton,gemeindetyp_code,gemeindetyp
0,6158,Vionnaz,23,Valais / Wallis,2,Intermediär
1,3023,Speicher,15,Appenzell Ausserrhoden,2,Intermediär
2,6011,Zwischbergen,23,Valais / Wallis,3,Ländlich
3,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,1,Städtisch
4,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,3,Ländlich


In [6]:
print("ElCom-Zeilen:", len(df_elcom_clean))
print("BFS-Zeilen:", len(df_bfs_clean))

print(
    "\nFehlende ElCom-Gemeindenummern:",
    df_elcom_clean["gemeindenummer"].isna().sum()
)

print(
    "Fehlende BFS-Typbezeichnungen:",
    df_bfs_clean["gemeindetyp"].isna().sum()
)

print(
    "\nDatentyp Gemeindenummer ElCom:",
    df_elcom_clean["gemeindenummer"].dtype
)

print(
    "Datentyp Gemeindenummer BFS:",
    df_bfs_clean["gemeindenummer"].dtype
)

print(
    "Datentyp Gesamttarif:",
    df_elcom_clean["gesamttarif"].dtype
)

print(
    "Fehlende Gesamttarife:",
    df_elcom_clean["gesamttarif"].isna().sum()
)

ElCom-Zeilen: 2268
BFS-Zeilen: 2110

Fehlende ElCom-Gemeindenummern: 0
Fehlende BFS-Typbezeichnungen: 0

Datentyp Gemeindenummer ElCom: Int64
Datentyp Gemeindenummer BFS: Int64
Datentyp Gesamttarif: float64
Fehlende Gesamttarife: 1


## 2.2 ElCom- und BFS-Daten verknüpfen

Wir ergänzen die ElCom-Tarife um Gemeindename, Kanton und Gemeindetyp
aus dem BFS-Datensatz.

Dazu verwenden wir einen Left Join über die Gemeindenummer.
Alle ElCom-Tarifzeilen bleiben zunächst erhalten.

Die Beziehung ist many-to-one:
Mehrere ElCom-Tarifzeilen können zu genau einer BFS-Gemeinde gehören.

Wir kennzeichnen nicht zugeordnete Zeilen und prüfen,
dass der Join die Anzahl der Tarifzeilen nicht verändert.

In [7]:
df_join = df_elcom_clean.merge(
    df_bfs_clean,
    on="gemeindenummer",
    how="left",
    validate="many_to_one",
    indicator=True
)

df_join.head()

,gemeinde,betreiber,gesamttarif,gemeindenummer,gemeindename,kanton_id,kanton,gemeindetyp_code,gemeindetyp,_merge
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,1,Aeugst am Albis,1.0,Zürich,3.0,Ländlich,both
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,10,Obfelden,1.0,Zürich,2.0,Intermediär,both
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,100,Stadel,1.0,Zürich,2.0,Intermediär,both
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1001,Doppleschwand,3.0,Luzern,3.0,Ländlich,both
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1002,Entlebuch,3.0,Luzern,3.0,Ländlich,both


In [8]:
print("ElCom-Zeilen vor dem Join:", len(df_elcom_clean))
print("Zeilen nach dem Join:", len(df_join))

print("\nZuordnung der Tarifzeilen:")
print(df_join["_merge"].value_counts())

print(
    "\nEindeutige zugeordnete Gemeinden:",
    df_join.loc[
        df_join["_merge"] == "both",
        "gemeindenummer"
    ].nunique()
)

# Der Left Join darf keine Tarifzeilen verlieren oder vervielfachen.
assert len(df_join) == len(df_elcom_clean), (
    "Die Zeilenanzahl hat sich durch den Join verändert."
)

ElCom-Zeilen vor dem Join: 2268
Zeilen nach dem Join: 2268

Zuordnung der Tarifzeilen:
_merge
both          2251
left_only       17
right_only       0
Name: count, dtype: int64

Eindeutige zugeordnete Gemeinden: 2105


In [9]:
nicht_zugeordnet = df_join.loc[
    df_join["_merge"] == "left_only",
    ["gemeindenummer", "gemeinde", "betreiber", "gesamttarif"]
].copy()

with pd.option_context("display.max_colwidth", None):
    display(nicht_zugeordnet)

,gemeindenummer,gemeinde,betreiber,gesamttarif
33,1057,https://ld.admin.ch/municipality/1057,https://energy.ld.admin.ch/elcom/electricityprice/operator/37,25.414
235,2016,https://ld.admin.ch/municipality/2016,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
238,2027,https://ld.admin.ch/municipality/2027,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
347,2278,https://ld.admin.ch/municipality/2278,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
401,2456,https://ld.admin.ch/municipality/2456,https://energy.ld.admin.ch/elcom/electricityprice/operator/813,38.086
923,3932,https://ld.admin.ch/municipality/3932,https://energy.ld.admin.ch/elcom/electricityprice/operator/617,29.917
1040,4122,https://ld.admin.ch/municipality/4122,https://energy.ld.admin.ch/elcom/electricityprice/operator/616,26.820
1074,4179,https://ld.admin.ch/municipality/4179,https://energy.ld.admin.ch/elcom/electricityprice/operator/356,24.047
1314,5064,https://ld.admin.ch/municipality/5064,https://energy.ld.admin.ch/elcom/electricityprice/operator/712,29.952
1320,5078,https://ld.admin.ch/municipality/5078,https://energy.ld.admin.ch/elcom/electricityprice/operator/712,29.952


### Nicht zugeordnete ElCom-Tarifzeilen

Beim Left Join können 17 ElCom-Tarifzeilen keiner Gemeinde
des BFS-Gemeindestands vom 01.01.2026 zugeordnet werden.

Die Zeilen besitzen einen Gesamttarif, erhalten durch den Join
aber keinen BFS-Gemeindenamen, Kanton oder Gemeindetyp.

Mögliche Ursachen sind unterschiedliche Gemeindestände oder
Gemeindemutationen. Diese Ursachen sind noch zu prüfen.

Die Zeilen bleiben vorerst erhalten. Eine automatische Zuordnung
über ähnliche Namen oder ein unbegründeter Ausschluss erfolgt nicht.

In [10]:
bfs_ohne_elcom = df_bfs_clean.loc[
    ~df_bfs_clean["gemeindenummer"].isin(
        df_elcom_clean["gemeindenummer"]
    ),
    [
        "gemeindenummer",
        "gemeindename",
        "kanton",
        "gemeindetyp"
    ]
].copy()

print("BFS-Gemeinden ohne ElCom-Tarifzeile:", len(bfs_ohne_elcom))

display(bfs_ohne_elcom)

BFS-Gemeinden ohne ElCom-Tarifzeile: 5


,gemeindenummer,gemeindename,kanton,gemeindetyp
848,2822,Augst,Basel-Landschaft,Städtisch
2093,64,Nürensdorf,Zürich,Intermediär
2102,5395,Lema,Ticino,Intermediär
2105,6831,Moutier,Jura,Intermediär
2106,2056,Fétigny-Ménières,Fribourg / Freiburg,Ländlich


In [11]:
join_bilanz = pd.DataFrame({
    "Prüfung": [
        "ElCom-Tarifzeilen vor dem Join",
        "Tarifzeilen nach dem Join",
        "Erfolgreich zugeordnete Tarifzeilen",
        "Nicht zugeordnete Tarifzeilen",
        "BFS-Gemeinden ohne ElCom-Tarifzeile"
    ],
    "Anzahl": [
        len(df_elcom_clean),
        len(df_join),
        (df_join["_merge"] == "both").sum(),
        (df_join["_merge"] == "left_only").sum(),
        len(bfs_ohne_elcom)
    ]
})

display(join_bilanz)

,Prüfung,Anzahl
0,ElCom-Tarifzeilen vor dem Join,2268
1,Tarifzeilen nach dem Join,2268
2,Erfolgreich zugeordnete Tarifzeilen,2251
3,Nicht zugeordnete Tarifzeilen,17
4,BFS-Gemeinden ohne ElCom-Tarifzeile,5


## 2.3 Gemeindenummern harmonisieren

Harmonisieren bedeutet hier, unterschiedliche Gemeindenummern
auf einen gemeinsamen Gemeindestand zu bringen.

### Bestätigter Fall: Moutier

Im ElCom-Abruf erscheint Moutier unter Gemeindenummer 700.
Zum 01.01.2026 wechselte Moutier vom Kanton Bern zum Kanton Jura
und erhielt die BFS-Gemeindenummer 6831.

Für den Join zum BFS-Gemeindestand 01.01.2026 verwenden wir
deshalb 6831. Die ursprüngliche ElCom-Gemeindenummer bleibt
zusätzlich erhalten.

Quelle:
https://www.agvchapp.bfs.admin.ch/de/mutations/3999

Weitere nicht zugeordnete Nummern werden separat geprüft.

In [12]:
df_elcom_harmonisiert = df_elcom_clean.copy()

# Amtlich bestätigte Änderung für Moutier.
nummern_mapping = {
    700: 6831
}

df_elcom_harmonisiert["gemeindenummer_2026"] = (
    df_elcom_harmonisiert["gemeindenummer"]
    .replace(nummern_mapping)
    .astype("Int64")
)

# Geänderte Nummern anzeigen.
display(
    df_elcom_harmonisiert.loc[
        df_elcom_harmonisiert["gemeindenummer"]
        != df_elcom_harmonisiert["gemeindenummer_2026"],
        ["gemeindenummer", "gemeindenummer_2026", "gesamttarif"]
    ]
)

,gemeindenummer,gemeindenummer_2026,gesamttarif
2085,700,6831,32.959


In [13]:
# Den BFS-Schlüssel passend benennen.
df_bfs_join = df_bfs_clean.rename(
    columns={"gemeindenummer": "gemeindenummer_2026"}
)

df_join_harmonisiert = df_elcom_harmonisiert.merge(
    df_bfs_join,
    on="gemeindenummer_2026",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Prüfen, ob Moutier nun korrekt zugeordnet ist.
display(
    df_join_harmonisiert.loc[
        df_join_harmonisiert["gemeindenummer"] == 700,
        [
            "gemeindenummer",
            "gemeindenummer_2026",
            "gemeindename",
            "kanton",
            "gemeindetyp",
            "gesamttarif",
            "_merge"
        ]
    ]
)

print("\nZuordnung nach der ersten Harmonisierung:")
print(df_join_harmonisiert["_merge"].value_counts())

assert len(df_join_harmonisiert) == len(df_elcom_clean)

,gemeindenummer,gemeindenummer_2026,gemeindename,kanton,gemeindetyp,gesamttarif,_merge
2085,700,6831,Moutier,Jura,Intermediär,32.959,both



Zuordnung nach der ersten Harmonisierung:
_merge
both          2252
left_only       16
right_only       0
Name: count, dtype: int64


### Weitere amtlich bestätigte Gemeindeänderungen

Wir ergänzen eine Zuordnungstabelle von früheren Gemeindenummern
zu den Gemeindenummern des Gemeindestands 01.01.2026.

Jede Zuordnung erhält einen Verweis auf die amtliche
BFS-Mutationsmeldung. Die ursprüngliche ElCom-Gemeindenummer
und die unveränderte Tarifangabe bleiben erhalten.

Bei Fusionen beziehen sich die ursprünglichen Tarifzeilen auf
frühere Gemeindegebiete. Durch die neue Zuordnung können mehrere
Zeilen dieselbe aktuelle Gemeinde-Betreiber-Kombination erhalten.
Diese Fälle prüfen wir vor einer Zusammenfassung separat.

In [14]:
# Spalten: ursprüngliche Nummer, Nummer zum Stand 2026,
# Nummer der amtlichen BFS-Mutationsmeldung.

zuordnungen = [
    (700, 6831, 3999),    # Moutier
    (1057, 1065, 3990),   # Honau -> Root
    (2016, 2056, 4000),   # Fétigny -> Fétigny-Ménières
    (2027, 2056, 4000),   # Ménières -> Fétigny-Ménières
    (2278, 2262, 4001),   # Ulmiz -> Gurmels
    (2456, 2465, 3985),   # Lüterswil-Gächliwil -> Buchegg
    (3932, 3901, 3989),   # Tschiertschen-Praden -> Chur
    (4122, 4095, 4003),   # Villnachern -> Brugg
    (4179, 4186, 3969),   # Ueken -> Herznach-Ueken
    (5064, 5073, 3995),   # Bodio -> Giornico
    (5078, 5079, 3996),   # Prato (Leventina) -> Quinto
    (5146, 5395, 3997),   # Astano -> Lema
    (5149, 5395, 3997),   # Bedigliora -> Lema
    (5181, 5395, 3997),   # Curio -> Lema
    (5200, 5395, 3997),   # Miglieglia -> Lema
    (5207, 5395, 3997),   # Novaggio -> Lema
    (6744, 6743, 3967)    # La Chaux-des-Breuleux -> Les Breuleux
]

df_nummernzuordnung = pd.DataFrame(
    zuordnungen,
    columns=[
        "gemeindenummer_original",
        "gemeindenummer_2026",
        "bfs_mutation"
    ]
)

# Quellen direkt in der Zuordnungstabelle dokumentieren.
df_nummernzuordnung["quelle"] = (
    "https://www.agvchapp.bfs.admin.ch/de/mutations/"
    + df_nummernzuordnung["bfs_mutation"].astype(str)
)

# Jede alte Nummer muss eindeutig zugeordnet sein.
assert df_nummernzuordnung["gemeindenummer_original"].is_unique

# Alle Zielnummern müssen im gewählten BFS-Bestand vorkommen.
assert df_nummernzuordnung["gemeindenummer_2026"].isin(
    df_bfs_clean["gemeindenummer"]
).all()

display(df_nummernzuordnung)

,gemeindenummer_original,gemeindenummer_2026,bfs_mutation,quelle
0,700,6831,3999,https://www.agvchapp.bfs.admin.ch/de/mutations...
1,1057,1065,3990,https://www.agvchapp.bfs.admin.ch/de/mutations...
2,2016,2056,4000,https://www.agvchapp.bfs.admin.ch/de/mutations...
3,2027,2056,4000,https://www.agvchapp.bfs.admin.ch/de/mutations...
4,2278,2262,4001,https://www.agvchapp.bfs.admin.ch/de/mutations...
5,2456,2465,3985,https://www.agvchapp.bfs.admin.ch/de/mutations...
6,3932,3901,3989,https://www.agvchapp.bfs.admin.ch/de/mutations...
7,4122,4095,4003,https://www.agvchapp.bfs.admin.ch/de/mutations...
8,4179,4186,3969,https://www.agvchapp.bfs.admin.ch/de/mutations...
9,5064,5073,3995,https://www.agvchapp.bfs.admin.ch/de/mutations...


In [15]:
nummern_mapping = df_nummernzuordnung.set_index(
    "gemeindenummer_original"
)["gemeindenummer_2026"].to_dict()

df_elcom_harmonisiert = df_elcom_clean.copy()

df_elcom_harmonisiert["gemeindenummer_2026"] = (
    df_elcom_harmonisiert["gemeindenummer"]
    .replace(nummern_mapping)
    .astype("Int64")
)

# Änderungen für jede Tarifzeile kennzeichnen.
df_elcom_harmonisiert["nummer_geaendert"] = (
    df_elcom_harmonisiert["gemeindenummer"]
    != df_elcom_harmonisiert["gemeindenummer_2026"]
)

df_bfs_join = df_bfs_clean.rename(
    columns={"gemeindenummer": "gemeindenummer_2026"}
)

df_join_harmonisiert = df_elcom_harmonisiert.merge(
    df_bfs_join,
    on="gemeindenummer_2026",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert len(df_join_harmonisiert) == len(df_elcom_clean)

print("Tarifzeilen nach Harmonisierung:",
      len(df_join_harmonisiert))

print("Tarifzeilen mit geänderter Nummer:",
      df_join_harmonisiert["nummer_geaendert"].sum())

print("\nZuordnung:")
print(df_join_harmonisiert["_merge"].value_counts())

Tarifzeilen nach Harmonisierung: 2268
Tarifzeilen mit geänderter Nummer: 17

Zuordnung:
_merge
both          2268
left_only        0
right_only       0
Name: count, dtype: int64


In [18]:
mehrfach_mask = df_join_harmonisiert.duplicated(
    subset=["gemeindenummer_2026", "betreiber"],
    keep=False
)

mehrfach_zugeordnet = df_join_harmonisiert.loc[
    mehrfach_mask,
    [
        "gemeindenummer",
        "gemeindenummer_2026",
        "gemeindename",
        "betreiber",
        "gesamttarif"
    ]
].sort_values(
    ["gemeindenummer_2026", "betreiber", "gemeindenummer"]
)

print(
    "Zeilen in mehrfach vorkommenden neuen "
    "Gemeinde-Betreiber-Kombinationen:",
    len(mehrfach_zugeordnet)
)

display(mehrfach_zugeordnet)

Zeilen in mehrfach vorkommenden neuen Gemeinde-Betreiber-Kombinationen: 23


,gemeindenummer,gemeindenummer_2026,gemeindename,betreiber,gesamttarif
33,1057,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
42,1065,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.153
235,2016,2056,Fétigny-Ménières,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
238,2027,2056,Fétigny-Ménières,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
337,2262,2262,Gurmels,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
347,2278,2262,Gurmels,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
401,2456,2465,Buchegg,https://energy.ld.admin.ch/elcom/electricitypr...,38.086
408,2465,2465,Buchegg,https://energy.ld.admin.ch/elcom/electricitypr...,38.086
917,3901,3901,Chur,https://energy.ld.admin.ch/elcom/electricitypr...,29.917
923,3932,3901,Chur,https://energy.ld.admin.ch/elcom/electricitypr...,29.917


## 2.4 Wiederholungen nach der Harmonisierung behandeln

Nach der Zuordnung auf den Gemeindestand 01.01.2026 gehören
23 Zeilen zu mehrfach vorkommenden Gemeinde-Betreiber-Kombinationen.

Wir unterscheiden:
- Identische Tarifwerte derselben aktuellen Gemeinde und desselben
  Betreibers werden für die weitere Analyse einmal berücksichtigt.
- Unterschiedliche Tarifwerte bleiben vorerst getrennt erhalten.

Damit vermeiden wir, dass ein identischer Tarif allein aufgrund
mehrerer früherer Gemeinden mehrfach gewichtet wird.

Beispielsweise unterscheiden sich die Tarifwerte für das frühere
Honau und das bisherige Root. Diese Werte werden nicht automatisch
gelöscht oder zu einem vermeintlich einheitlichen Tarif erklärt.

Die vollständigen ursprünglichen Zuordnungen bleiben in der
harmonisierten Ausgangstabelle erhalten.

In [19]:
# Benötigte Analysefelder auswählen.
# Die vollständige Herkunft bleibt in df_join_harmonisiert erhalten.
df_tarife_eindeutig = (
    df_join_harmonisiert[
        [
            "gemeindenummer_2026",
            "gemeindename",
            "kanton_id",
            "kanton",
            "gemeindetyp_code",
            "gemeindetyp",
            "betreiber",
            "gesamttarif",
            "_merge"
        ]
    ]
    .drop_duplicates(
        subset=[
            "gemeindenummer_2026",
            "betreiber",
            "gesamttarif"
        ]
    )
    .copy()
)

print("Zeilen vor der Zusammenfassung:",
      len(df_join_harmonisiert))

print("Zeilen nach der Zusammenfassung:",
      len(df_tarife_eindeutig))

print("Zusammengefasste Wiederholungen:",
      len(df_join_harmonisiert) - len(df_tarife_eindeutig))

print("\nFehlende Gesamttarife:",
      df_tarife_eindeutig["gesamttarif"].isna().sum())

print("\nBFS-Zuordnung:")
print(df_tarife_eindeutig["_merge"].value_counts())

Zeilen vor der Zusammenfassung: 2268
Zeilen nach der Zusammenfassung: 2256
Zusammengefasste Wiederholungen: 12

Fehlende Gesamttarife: 1

BFS-Zuordnung:
_merge
both          2256
left_only        0
right_only       0
Name: count, dtype: int64


In [20]:
display(
    df_tarife_eindeutig.loc[
        df_tarife_eindeutig["gemeindenummer_2026"] == 1065,
        [
            "gemeindenummer_2026",
            "gemeindename",
            "betreiber",
            "gesamttarif"
        ]
    ]
)

,gemeindenummer_2026,gemeindename,betreiber,gesamttarif
33,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
42,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.153


## 2.5 Fehlenden Gesamttarif behandeln

Nach der Harmonisierung und Zusammenfassung identischer
Gemeinde-Betreiber-Tarif-Kombinationen verbleiben 2’256 Tarifzeilen.
Alle Zeilen sind einer BFS-Gemeinde zugeordnet.

Bei einer Zeile fehlt der Gesamttarif. Diese Zeile wird aus
der Tarifanalyse ausgeschlossen, weil die Zielvariable fehlt.

Wir ersetzen den fehlenden Tarif weder durch null noch durch
einen geschätzten Wert. Die ausgeschlossene Zeile bleibt
in einer separaten Tabelle und in den Rohdaten erhalten.

Anschliessend prüfen wir, ob der Ausschluss dazu führt, dass
eine Gemeinde überhaupt keinen gültigen Tarif mehr besitzt.

In [21]:
# Zeile ohne Gesamttarif separat dokumentieren.
df_ausgeschlossen = df_tarife_eindeutig.loc[
    df_tarife_eindeutig["gesamttarif"].isna()
].copy()

df_ausgeschlossen["ausschlussgrund"] = "Gesamttarif fehlt"

# Gültige Tarifzeilen für die weitere Analyse.
df_tarife_analyse = df_tarife_eindeutig.loc[
    df_tarife_eindeutig["gesamttarif"].notna()
].copy()

print("Ausgeschlossene Tarifzeilen:", len(df_ausgeschlossen))
print("Tarifzeilen für die Analyse:", len(df_tarife_analyse))

display(df_ausgeschlossen)

Ausgeschlossene Tarifzeilen: 1
Tarifzeilen für die Analyse: 2255


,gemeindenummer_2026,gemeindename,kanton_id,kanton,gemeindetyp_code,gemeindetyp,betreiber,gesamttarif,_merge,ausschlussgrund
699,3218,Tübach,17,St. Gallen,1,Städtisch,https://energy.ld.admin.ch/elcom/electricitypr...,NaN,both,Gesamttarif fehlt


In [22]:
gemeinden_mit_tarif = (
    df_tarife_analyse["gemeindenummer_2026"].nunique()
)

df_gemeinden_ohne_tarif = df_bfs_clean.loc[
    ~df_bfs_clean["gemeindenummer"].isin(
        df_tarife_analyse["gemeindenummer_2026"]
    ),
    [
        "gemeindenummer",
        "gemeindename",
        "kanton",
        "gemeindetyp"
    ]
].copy()

print("BFS-Gemeinden insgesamt:", len(df_bfs_clean))
print("Gemeinden mit gültigem Tarif:", gemeinden_mit_tarif)
print("Gemeinden ohne gültigen Tarif:",
      len(df_gemeinden_ohne_tarif))

display(df_gemeinden_ohne_tarif)

# Beide Gruppen müssen zusammen den BFS-Gesamtbestand ergeben.
assert (
    gemeinden_mit_tarif + len(df_gemeinden_ohne_tarif)
    == len(df_bfs_clean)
)

# Keine Zeile darf bei der Aufteilung verloren gehen.
assert (
    len(df_tarife_analyse) + len(df_ausgeschlossen)
    == len(df_tarife_eindeutig)
)

BFS-Gemeinden insgesamt: 2110
Gemeinden mit gültigem Tarif: 2108
Gemeinden ohne gültigen Tarif: 2


,gemeindenummer,gemeindename,kanton,gemeindetyp
848,2822,Augst,Basel-Landschaft,Städtisch
2093,64,Nürensdorf,Zürich,Intermediär


In [23]:
bereinigungsbilanz = pd.DataFrame({
    "Schritt": [
        "ElCom-Tarifzeilen im Rohdatensatz",
        "Tarifzeilen mit harmonisierter Gemeindenummer",
        "Zusammengefasste identische Tarifkombinationen",
        "Tarifzeilen nach Zusammenfassung",
        "Ausgeschlossene Zeilen ohne Gesamttarif",
        "Verbleibende Analysezeilen",
        "BFS-Gemeinden insgesamt",
        "Gemeinden mit gültigem Tarif",
        "Gemeinden ohne gültigen Tarif"
    ],
    "Anzahl": [
        len(df_elcom_clean),
        df_join_harmonisiert["nummer_geaendert"].sum(),
        len(df_join_harmonisiert) - len(df_tarife_eindeutig),
        len(df_tarife_eindeutig),
        len(df_ausgeschlossen),
        len(df_tarife_analyse),
        len(df_bfs_clean),
        df_tarife_analyse["gemeindenummer_2026"].nunique(),
        len(df_gemeinden_ohne_tarif)
    ]
})

display(bereinigungsbilanz)

,Schritt,Anzahl
0,ElCom-Tarifzeilen im Rohdatensatz,2268
1,Tarifzeilen mit harmonisierter Gemeindenummer,17
2,Zusammengefasste identische Tarifkombinationen,12
3,Tarifzeilen nach Zusammenfassung,2256
4,Ausgeschlossene Zeilen ohne Gesamttarif,1
5,Verbleibende Analysezeilen,2255
6,BFS-Gemeinden insgesamt,2110
7,Gemeinden mit gültigem Tarif,2108
8,Gemeinden ohne gültigen Tarif,2


## 2.6 Ergebnis der Datenaufbereitung und Integration

Die ElCom-Rohdaten enthalten 2’268 Tarifzeilen.

Beim ursprünglichen Join konnten 17 Tarifzeilen keiner aktuellen
BFS-Gemeinde zugeordnet werden. Nach Harmonisierung der
Gemeindenummern anhand amtlicher BFS-Mutationsmeldungen sind
alle Tarifzeilen dem Gemeindestand 01.01.2026 zugeordnet.

Anschliessend wurden 12 Wiederholungen gleicher Kombinationen
aus aktueller Gemeinde, Betreiber und Gesamttarif zusammengefasst.
Unterschiedliche Tarifwerte derselben Gemeinde und desselben
Betreibers blieben erhalten.

Eine weitere Zeile wurde wegen eines fehlenden Gesamttarifs
aus der Tarifanalyse ausgeschlossen und separat dokumentiert.

Der aufbereitete Datensatz enthält:
- 2’255 Tarifzeilen,
- 2’108 Gemeinden mit mindestens einem gültigen Tarif.

Für Augst und Nürensdorf liegt im abgefragten Bestand keine
passende Tarifzeile vor. Beide Gemeinden bleiben in der
BFS-Stammdatentabelle erhalten, gehen aber vorerst nicht
in die Tarifanalyse ein. Fehlende Tarife werden nicht geschätzt.

Die ursprünglichen Rohdaten und die Herkunft der harmonisierten
Tarifzeilen bleiben nachvollziehbar erhalten.

Die Analysetabelle enthält weiterhin mehrere Tarifzeilen je
Gemeinde. Vor dem statistischen Gemeindevergleich legen wir
fest, wie daraus ein Gemeindewert gebildet wird.

In [25]:
processed_ordner = projektordner / "data" / "processed"
processed_ordner.mkdir(parents=True, exist_ok=True)

tabellen = {
    "tarife_bereinigt.csv": df_tarife_analyse,
    "bfs_gemeinden_2026.csv": df_bfs_clean,
    "gemeindenummern_zuordnung.csv": df_nummernzuordnung,
    "tarife_herkunft.csv": df_join_harmonisiert,
    "ausgeschlossene_tarife.csv": df_ausgeschlossen,
    "gemeinden_ohne_tarif.csv": df_gemeinden_ohne_tarif,
    "bereinigungsbilanz.csv": bereinigungsbilanz
}

for dateiname, tabelle in tabellen.items():
    tabelle.to_csv(
        processed_ordner / dateiname,
        index=False,
        encoding="utf-8"
    )
    print("Gespeichert:", dateiname)

Gespeichert: tarife_bereinigt.csv
Gespeichert: bfs_gemeinden_2026.csv
Gespeichert: gemeindenummern_zuordnung.csv
Gespeichert: tarife_herkunft.csv
Gespeichert: ausgeschlossene_tarife.csv
Gespeichert: gemeinden_ohne_tarif.csv
Gespeichert: bereinigungsbilanz.csv
